# ResNet VAE from Scratch (Colab)

Companion notebook for the [Autoencoder Architecture](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html) tutorial. We implement only the **ResNet VAE** — the most complete architecture covering ResNet blocks, self-attention, variational inference, and spatial latents.

See the full tutorial for theory, derivations, and progressive architecture comparisons.

In [ ]:
#@title Install dependencies
try:
    import torch, plotly, rich, sklearn
except ImportError:
    !pip install -q torch torchvision plotly ipywidgets rich scikit-learn tensorboard

### Hyperparameters

| Parameter | What it controls |
|-----------|------------------|
| `BATCH_SIZE` | Images per training step. Larger = faster but needs more GPU memory |
| `EPOCHS` | Full passes over the 60k training images. 100 is a good default |
| `KL_WEIGHT` | Balance between sharp reconstructions (lower) and smooth latent space (higher). See [Why the β weight?](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html#from-elbo-to-code-the-actual-vae-loss) |
| `LEARNING_RATE` | Adam optimizer step size. Too high = unstable, too low = slow |
| `Z_CHANNELS` | Channels in the spatial latent (z_ch × 7 × 7). More channels = more capacity but harder to regularize |

In [ ]:
#@title Configuration { run: "auto" }

BATCH_SIZE = 256  #@param {type:"integer"}  # Images per training step
EPOCHS = 100  #@param {type:"integer"}  # Full passes over the dataset
KL_WEIGHT = 0.0005  #@param {type:"number"}  # Beta: reconstruction vs latent smoothness
LEARNING_RATE = 1e-3  #@param {type:"number"}  # Adam optimizer step size
Z_CHANNELS = 1  #@param {type:"integer"}  # Latent channels (spatial: z_ch x 7 x 7)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from rich.console import Console
from rich.table import Table

console = Console()
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
console.print(f"[bold green]Device:[/bold green] {DEVICE}")

writer = SummaryWriter("runs/resnet_vae")

## 1. Dataset: FashionMNIST

See [Why Compress Images?](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html#why-compress-images) in the full tutorial.

In [ ]:
#@title Load FashionMNIST

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

t = Table(title="FashionMNIST Dataset")
t.add_column("Split", style="cyan")
t.add_column("Samples", style="green")
t.add_column("Image Size", style="magenta")
t.add_column("Classes", style="dim")
t.add_row("Train", str(len(train_dataset)), "28 \u00d7 28 \u00d7 1", "10")
t.add_row("Test", str(len(test_dataset)), "28 \u00d7 28 \u00d7 1", "10")
console.print(t)

# Sample grid
sample_images, sample_labels = next(iter(test_loader))
indices = []
for c in range(10):
    class_idx = (sample_labels == c).nonzero(as_tuple=True)[0][:2]
    indices.extend(class_idx.tolist())
indices = indices[:20]

fig = make_subplots(
    rows=2, cols=10,
    subplot_titles=[CLASS_NAMES[sample_labels[i].item()] for i in indices],
    vertical_spacing=0.08, horizontal_spacing=0.02,
)
for pos, idx in enumerate(indices):
    row, col = pos // 10 + 1, pos % 10 + 1
    img = sample_images[idx].squeeze().numpy()
    fig.add_trace(
        go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False,
                   hovertemplate="pixel (%{x}, %{y}): %{z:.2f}<extra></extra>"),
        row=row, col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, row=row, col=col)
fig.update_layout(
    title_text="FashionMNIST \u2014 Sample Grid (2 per class)",
    height=320, width=900, margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

# Grab one image per class for interpolation later
class_images = {}
for images, labels in test_loader:
    for c in range(10):
        if c not in class_images:
            match = (labels == c).nonzero(as_tuple=True)[0]
            if len(match) > 0:
                class_images[c] = images[match[0]]
    if len(class_images) == 10:
        break

## 2. Architecture

See [Scaling Up — A ResNet VAE](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html#scaling-up-a-resnet-vae) for the full architecture walkthrough.

In [ ]:
#@title Building blocks: ResNetBlock & SelfAttention

class ResNetBlock(nn.Module):
    """Residual block: GroupNorm \u2192 SiLU \u2192 Conv \u2192 GroupNorm \u2192 SiLU \u2192 Conv + skip."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.GroupNorm(min(8, in_ch), in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.net(x) + self.skip(x)


class SelfAttention(nn.Module):
    """Single-head self-attention over spatial positions."""

    def __init__(self, ch: int):
        super().__init__()
        self.norm = nn.GroupNorm(min(8, ch), ch)
        self.q = nn.Conv2d(ch, ch, 1)
        self.k = nn.Conv2d(ch, ch, 1)
        self.v = nn.Conv2d(ch, ch, 1)
        self.proj = nn.Conv2d(ch, ch, 1)
        self.scale = ch ** -0.5

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        q = self.q(h).view(B, C, -1)
        k = self.k(h).view(B, C, -1)
        v = self.v(h).view(B, C, -1)
        attn = (q.transpose(1, 2) @ k) * self.scale
        attn = attn.softmax(dim=-1)
        out = (v @ attn.transpose(1, 2)).view(B, C, H, W)
        return x + self.proj(out)

With the building blocks defined, we assemble the full encoder → bottleneck → decoder pipeline.

In [ ]:
#@title ResNet VAE model

class ResNetEncoder(nn.Module):
    """Encoder: 1\u00d728\u00d728 \u2192 spatial latent (z_ch \u00d7 7 \u00d7 7)."""

    def __init__(self, z_channels: int = 4):
        super().__init__()
        self.conv_in = nn.Conv2d(1, 32, 3, padding=1)

        self.down1 = nn.Sequential(
            ResNetBlock(32, 64), ResNetBlock(64, 64),
            nn.Conv2d(64, 64, 3, stride=2, padding=1),
        )
        self.down2 = nn.Sequential(
            ResNetBlock(64, 128), ResNetBlock(128, 128),
            nn.Conv2d(128, 128, 3, stride=2, padding=1),
        )
        self.mid = nn.Sequential(
            ResNetBlock(128, 128), SelfAttention(128), ResNetBlock(128, 128),
        )
        self.norm_out = nn.GroupNorm(8, 128)
        self.conv_out = nn.Conv2d(128, 2 * z_channels, 3, padding=1)

    def forward(self, x):
        h = self.conv_in(x)
        h = self.down1(h)
        h = self.down2(h)
        h = self.mid(h)
        h = F.silu(self.norm_out(h))
        return self.conv_out(h)


class ResNetDecoder(nn.Module):
    """Decoder: spatial latent (z_ch \u00d7 7 \u00d7 7) \u2192 1\u00d728\u00d728."""

    def __init__(self, z_channels: int = 4):
        super().__init__()
        self.conv_in = nn.Conv2d(z_channels, 128, 3, padding=1)

        self.mid = nn.Sequential(
            ResNetBlock(128, 128), SelfAttention(128), ResNetBlock(128, 128),
        )
        self.up2 = nn.Sequential(
            ResNetBlock(128, 128), ResNetBlock(128, 128), ResNetBlock(128, 64),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(64, 64, 3, padding=1),
        )
        self.up1 = nn.Sequential(
            ResNetBlock(64, 64), ResNetBlock(64, 64), ResNetBlock(64, 32),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(32, 32, 3, padding=1),
        )
        self.norm_out = nn.GroupNorm(8, 32)
        self.conv_out = nn.Conv2d(32, 1, 3, padding=1)

    def forward(self, z):
        h = self.conv_in(z)
        h = self.mid(h)
        h = self.up2(h)
        h = self.up1(h)
        h = F.silu(self.norm_out(h))
        return torch.sigmoid(self.conv_out(h))


class ResNetVAE(nn.Module):
    """ResNet-based VAE with spatial latent: 1\u00d728\u00d728 \u2192 z_ch\u00d77\u00d77 \u2192 1\u00d728\u00d728."""

    def __init__(self, z_channels: int = 4):
        super().__init__()
        self.encoder = ResNetEncoder(z_channels)
        self.decoder = ResNetDecoder(z_channels)
        self.z_channels = z_channels

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    def forward(self, x):
        h = self.encoder(x)
        mu, log_var = h.chunk(2, dim=1)
        z = self.reparameterize(mu, log_var)
        x_hat = self.decoder(z)
        return x_hat, mu, log_var, z


model = ResNetVAE(z_channels=Z_CHANNELS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

t = Table(title="ResNet VAE Architecture")
t.add_column("Component", style="cyan")
t.add_column("Layer", style="magenta")
t.add_column("Output Shape", style="green")
for name, layer, shape in [
    ("Encoder", "conv_in",                       "32\u00d728\u00d728"),
    ("",        "2\u00d7 ResBlock(32\u219264) + \u2193stride",  "64\u00d714\u00d714"),
    ("",        "2\u00d7 ResBlock(64\u2192128) + \u2193stride",  "128\u00d77\u00d77"),
    ("",        "ResBlock + SelfAttn + ResBlock",  "128\u00d77\u00d77"),
    ("",        "GN + SiLU + Conv \u2192 \u03bc, log \u03c3\u00b2",   f"2\u00d7({Z_CHANNELS}\u00d77\u00d77)"),
    ("Latent",  "Reparameterize",                  f"{Z_CHANNELS}\u00d77\u00d77 = {Z_CHANNELS * 49} values"),
    ("Decoder", "conv_in",                        "128\u00d77\u00d77"),
    ("",        "ResBlock + SelfAttn + ResBlock",  "128\u00d77\u00d77"),
    ("",        "3\u00d7 ResBlock(128\u219264) + \u2191nearest",  "64\u00d714\u00d714"),
    ("",        "3\u00d7 ResBlock(64\u219232) + \u2191nearest",   "32\u00d728\u00d728"),
    ("",        "GN + SiLU + Conv + Sigmoid",      "1\u00d728\u00d728"),
]:
    t.add_row(name, layer, shape)
console.print(t)

total_params = sum(p.numel() for p in model.parameters())
console.print(f"\n[bold]Total parameters:[/bold] {total_params:,}")
console.print(f"[bold]Latent shape:[/bold] {Z_CHANNELS}\u00d77\u00d77 = {Z_CHANNELS * 49} values")

The encoder compresses 1×28×28 → 1×7×7 (49 latent values), and the decoder reconstructs from that bottleneck alone — no skip connections.

## 3. Training

The loss is the negative ELBO: MSE reconstruction + \u03b2 \u00b7 KL divergence. See [From ELBO to code](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html#from-elbo-to-code-the-actual-vae-loss) for the derivation.

In [ ]:
#@title TensorBoard

%load_ext tensorboard
%tensorboard --logdir runs

TensorBoard will update live as training runs. The **Reconstructions** tab shows how output quality improves over epochs.

In [ ]:
from torchvision.utils import make_grid

criterion = nn.MSELoss()
# Fixed samples for consistent TensorBoard image tracking
fixed_test_imgs = sample_images[:8].to(DEVICE)

for epoch in range(EPOCHS):
    model.train()
    epoch_total, epoch_recon, epoch_kl = 0.0, 0.0, 0.0

    for images, _ in train_loader:
        images = images.to(DEVICE)
        x_hat, mu, log_var, _ = model(images)

        recon = criterion(x_hat, images)
        kl = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
        loss = recon + KL_WEIGHT * kl

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        b = images.size(0)
        epoch_total += loss.item() * b
        epoch_recon += recon.item() * b
        epoch_kl += kl.item() * b

    n = len(train_dataset)
    avg_total = epoch_total / n
    avg_recon = epoch_recon / n
    avg_kl = epoch_kl / n

    # TensorBoard: scalars
    writer.add_scalar("Loss/total", avg_total, epoch)
    writer.add_scalar("Loss/reconstruction", avg_recon, epoch)
    writer.add_scalar("Loss/kl_divergence", avg_kl, epoch)

    # TensorBoard: reconstructions (every 5 epochs + first and last)
    if epoch == 0 or (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            recon_imgs, _, _, _ = model(fixed_test_imgs)
        comparison = torch.cat([fixed_test_imgs, recon_imgs])
        writer.add_image("Reconstructions", make_grid(comparison, nrow=8), epoch)

    if (epoch + 1) % 10 == 0:
        console.print(
            f"Epoch {epoch+1:3d}/{EPOCHS}  "
            f"total={avg_total:.6f}  recon={avg_recon:.6f}  kl={avg_kl:.4f}"
        )

writer.flush()
console.print("[bold green]Training complete![/bold green]")


## 4. Results

See [Reconstructions and generation](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html#reconstructions-and-generation) for comparison with simpler architectures.

In [ ]:
#@title Reconstruction comparison

model.eval()

with torch.no_grad():
    imgs = sample_images[:10].to(DEVICE)
    recon, _, _, _ = model(imgs)
    recon = recon.cpu()

fig = make_subplots(
    rows=2, cols=10, vertical_spacing=0.02, horizontal_spacing=0.01,
    row_titles=["Original", "ResNet VAE"],
)
for r, row_imgs in enumerate([sample_images[:10], recon]):
    for c in range(10):
        img = row_imgs[c].squeeze().numpy()
        fig.add_trace(
            go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False,
                       hovertemplate="(%{x},%{y}): %{z:.2f}<extra></extra>"),
            row=r + 1, col=c + 1,
        )
        fig.update_xaxes(showticklabels=False, row=r + 1, col=c + 1)
        fig.update_yaxes(showticklabels=False, row=r + 1, col=c + 1)

fig.update_layout(
    title_text="Reconstructions \u2014 Original vs ResNet VAE",
    height=250, width=900, margin=dict(t=40, b=10, l=80, r=10),
)
fig.show()

test_mse = criterion(recon, sample_images[:10]).item()
console.print(f"[bold]Test MSE:[/bold] {test_mse:.6f}")

Since the VAE’s latent space is regularized, we can also **generate new images** by sampling directly from the prior.

In [ ]:
#@title Generate from prior

# Generate new images by sampling from the prior
model.eval()
with torch.no_grad():
    z_sample = torch.randn(20, model.z_channels, 7, 7, device=DEVICE)
    generated = model.decoder(z_sample).cpu()

fig = make_subplots(rows=2, cols=10, vertical_spacing=0.04, horizontal_spacing=0.02)
for i in range(20):
    row, col = i // 10 + 1, i % 10 + 1
    img = generated[i].squeeze().numpy()
    fig.add_trace(
        go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False,
                   hovertemplate="(%{x},%{y}): %{z:.2f}<extra></extra>"),
        row=row, col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, row=row, col=col)

fig.update_layout(
    title_text="ResNet VAE \u2014 Generated Samples (z ~ N(0, I))",
    height=250, width=900, margin=dict(t=40, b=10, l=10, r=10),
)
fig.show()

## 5. Latent Space

See [Latent interpolation](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html#sec-latent-interpolation) for why smooth interpolation matters.

In [ ]:
#@title t-SNE projection of the latent space

from sklearn.manifold import TSNE

model.eval()
all_mu, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        h = model.encoder(images.to(DEVICE))
        mu, _ = h.chunk(2, dim=1)
        all_mu.append(mu.cpu().view(mu.size(0), -1))
        all_labels.append(labels)

all_mu = torch.cat(all_mu).numpy()
all_labels = torch.cat(all_labels).numpy()

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
z_2d = tsne.fit_transform(all_mu)

PALETTE = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990",
]

fig = go.Figure()
for c in range(10):
    mask = all_labels == c
    fig.add_trace(go.Scattergl(
        x=z_2d[mask, 0], y=z_2d[mask, 1],
        mode="markers", marker=dict(size=3, color=PALETTE[c], opacity=0.6),
        name=CLASS_NAMES[c],
    ))
fig.update_layout(
    title="t-SNE of ResNet VAE Latent Space",
    height=500, width=700, template="plotly_white",
    legend=dict(itemsizing="constant"),
)
fig.show()

The clusters confirm the latent space is well-organized. Let’s test its **smoothness** by interpolating between class pairs.

In [ ]:
#@title Cross-class interpolation

# Cross-class latent interpolation
model.eval()

interp_pairs = [
    (1, 7),  # Trouser \u2192 Sneaker
    (8, 5),  # Bag \u2192 Sandal
    (0, 9),  # T-shirt \u2192 Ankle boot
    (2, 8),  # Pullover \u2192 Bag
    (3, 7),  # Dress \u2192 Sneaker
]
n_steps = 10

fig = make_subplots(
    rows=len(interp_pairs), cols=n_steps,
    vertical_spacing=0.03, horizontal_spacing=0.01,
    row_titles=[f"{CLASS_NAMES[a]} \u2192 {CLASS_NAMES[b]}" for a, b in interp_pairs],
    column_titles=[f"{t:.0%}" for t in np.linspace(0, 1, n_steps)],
)

with torch.no_grad():
    for row, (cls_a, cls_b) in enumerate(interp_pairs):
        img_a = class_images[cls_a].unsqueeze(0).to(DEVICE)
        img_b = class_images[cls_b].unsqueeze(0).to(DEVICE)

        h_a = model.encoder(img_a)
        mu_a, _ = h_a.chunk(2, dim=1)
        h_b = model.encoder(img_b)
        mu_b, _ = h_b.chunk(2, dim=1)

        for col, t in enumerate(np.linspace(0, 1, n_steps)):
            z_t = (1 - t) * mu_a + t * mu_b
            img_t = model.decoder(z_t).squeeze().cpu().numpy()
            fig.add_trace(
                go.Heatmap(z=img_t[::-1], colorscale="Gray_r", showscale=False,
                           hovertemplate="(%{x},%{y}): %{z:.2f}<extra></extra>"),
                row=row + 1, col=col + 1,
            )
            fig.update_xaxes(showticklabels=False, row=row + 1, col=col + 1)
            fig.update_yaxes(showticklabels=False, row=row + 1, col=col + 1)

fig.update_layout(
    title_text="Cross-Class Interpolation (ResNet VAE)",
    height=150 * len(interp_pairs), width=900,
    margin=dict(t=60, b=10, l=120, r=10),
)
fig.show()

## 6. Interactive Latent Interpolation

Use the dropdowns to pick two classes and the slider to walk between them in latent space.

In [ ]:
#@title Interactive latent interpolation

# Enable third-party widgets (needed for FigureWidget in Colab)
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not running in Colab

import ipywidgets as widgets
from IPython.display import display

model.eval()

N_PAIRS = 4  # number of simultaneous interpolation rows
IMG_SIZE = 28
PAD = 2  # pixels between images

# Collect all test images grouped by class
class_image_pools = {c: [] for c in range(10)}
for images, labels in test_loader:
    for c in range(10):
        mask = (labels == c).nonzero(as_tuple=True)[0]
        for idx in mask:
            class_image_pools[c].append(images[idx])

# Current encoded means for N_PAIRS pairs
state = {"mus_a": [], "mus_b": []}


def encode_random_pairs():
    cls_a = class_a_dropdown.value
    cls_b = class_b_dropdown.value
    state["mus_a"], state["mus_b"] = [], []
    with torch.no_grad():
        idxs_a = np.random.choice(len(class_image_pools[cls_a]), N_PAIRS, replace=False)
        idxs_b = np.random.choice(len(class_image_pools[cls_b]), N_PAIRS, replace=False)
        for idx_a, idx_b in zip(idxs_a, idxs_b):
            h_a = model.encoder(class_image_pools[cls_a][idx_a].unsqueeze(0).to(DEVICE))
            mu_a, _ = h_a.chunk(2, dim=1)
            state["mus_a"].append(mu_a)
            h_b = model.encoder(class_image_pools[cls_b][idx_b].unsqueeze(0).to(DEVICE))
            mu_b, _ = h_b.chunk(2, dim=1)
            state["mus_b"].append(mu_b)


def build_grid(t_val):
    """Decode all pairs at interpolation t and tile into one image array."""
    imgs = []
    with torch.no_grad():
        for i in range(N_PAIRS):
            z_t = (1 - t_val) * state["mus_a"][i] + t_val * state["mus_b"][i]
            img = model.decoder(z_t).squeeze().cpu().numpy()
            imgs.append(img)
    # Tile horizontally with padding
    grid_w = N_PAIRS * IMG_SIZE + (N_PAIRS - 1) * PAD
    grid = np.ones((IMG_SIZE, grid_w)) * 0.5  # gray padding
    for i, img in enumerate(imgs):
        x0 = i * (IMG_SIZE + PAD)
        grid[:, x0:x0 + IMG_SIZE] = img
    return grid


class_options = {CLASS_NAMES[i]: i for i in range(10)}

class_a_dropdown = widgets.Dropdown(
    options=class_options, value=1, description="Class A:",
)
class_b_dropdown = widgets.Dropdown(
    options=class_options, value=7, description="Class B:",
)
t_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description="t:", continuous_update=True,
    style={"description_width": "30px"},
    layout=widgets.Layout(width="500px"),
)
randomize_btn = widgets.Button(
    description="Random pairs",
    button_style="info",
    icon="random",
)

# Initial encoding and render
encode_random_pairs()
init_grid = build_grid(0.5)

fig_widget = go.FigureWidget(
    data=[go.Heatmap(
        z=init_grid[::-1], colorscale="Gray_r", showscale=False,
        hovertemplate="(%{x},%{y}): %{z:.2f}<extra></extra>",
    )],
    layout=dict(
        title="Trouser \u2192 Sneaker  (t=0.50)",
        height=250, width=200 * N_PAIRS,
        xaxis=dict(showticklabels=False, scaleanchor="y"),
        yaxis=dict(showticklabels=False),
        margin=dict(t=40, b=10, l=10, r=10),
    ),
)


def render(change=None):
    t = t_slider.value
    cls_a = class_a_dropdown.value
    cls_b = class_b_dropdown.value
    grid = build_grid(t)
    with fig_widget.batch_update():
        fig_widget.data[0].z = grid[::-1]
        fig_widget.layout.title.text = (
            f"{CLASS_NAMES[cls_a]} \u2192 {CLASS_NAMES[cls_b]}  (t={t:.2f})"
        )


def on_randomize(btn):
    encode_random_pairs()
    render()


def on_class_change(change):
    encode_random_pairs()
    render()


class_a_dropdown.observe(on_class_change, names="value")
class_b_dropdown.observe(on_class_change, names="value")
t_slider.observe(render, names="value")
randomize_btn.on_click(on_randomize)

display(widgets.VBox([
    widgets.HBox([class_a_dropdown, class_b_dropdown, randomize_btn]),
    t_slider,
    fig_widget,
]))


---

**Full tutorial:** [Autoencoder Architecture](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html) | **Blog:** [Overfitting Club](https://overfitting.club)